In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

In [ ]:
#ruta
SPLIT_PATH = "/content/drive/MyDrive/tesis/split_1.npz"

LABEL_ORDER = [1, 2, 3, 4, 5]
CLASS_NAMES = ["Hepatocito", "Estrellada", "Kupffer", "Endotelial", "Otras"]

RANDOM_STATE = 42

# Carpeta
OUTPUT_DIR = "/content/drive/MyDrive/tesis/linearsvc_pipeline_pca_macro_1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# output
RESULTS_CSV_PATH = os.path.join(OUTPUT_DIR, "linearsvc_hyperparameter_search_results.csv")
BEST_PARAMS_TXT_PATH = os.path.join(OUTPUT_DIR, "best_params_linearsvc.txt")
MODEL_PATH = os.path.join(OUTPUT_DIR, "linearsvc_pipeline.pkl")

METRICS_CSV_PATH = os.path.join(OUTPUT_DIR, "metrics_summary_linearsvc.csv")
METRICS_TXT_PATH = os.path.join(OUTPUT_DIR, "metrics_report_linearsvc.txt")
METRICS_PNG_PATH = os.path.join(OUTPUT_DIR, "metrics_summary_linearsvc.png")

CM_VAL_PATH = os.path.join(OUTPUT_DIR, "confusion_matrix_validacion_linearsvc.png")
CM_TEST_PATH = os.path.join(OUTPUT_DIR, "confusion_matrix_test_linearsvc.png")

print("Carpeta de salida:", OUTPUT_DIR)

In [ ]:
data = np.load(SPLIT_PATH)

X_train = data["X_train"]
y_train = data["y_train"]
X_val   = data["X_val"]
y_val   = data["y_val"]
X_test  = data["X_test"]
y_test  = data["y_test"]

print("\n==============================")
print("Shapes cargados desde split.npz")
print("==============================")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nClases en train:", np.unique(y_train))
print("Clases en val  :", np.unique(y_val))
print("Clases en test :", np.unique(y_test))

In [ ]:
def evaluate_model(y_true, y_pred, split_name):
    acc = accuracy_score(y_true, y_pred)

    precision_w = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall_w    = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_w        = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    precision_m = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_m    = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_m        = f1_score(y_true, y_pred, average="macro", zero_division=0)

    bal_acc = balanced_accuracy_score(y_true, y_pred)

    report_txt = classification_report(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )

    print("\n" + "=" * 70)
    print(f"RESULTADOS - {split_name.upper()}")
    print("=" * 70)
    print(f"Accuracy           : {acc:.4f}")
    print(f"Precision weighted : {precision_w:.4f}")
    print(f"Recall weighted    : {recall_w:.4f}")
    print(f"F1-score weighted  : {f1_w:.4f}")
    print(f"Precision macro    : {precision_m:.4f}")
    print(f"Recall macro       : {recall_m:.4f}")
    print(f"F1-score macro     : {f1_m:.4f}")
    print(f"Balanced Accuracy  : {bal_acc:.4f}")

    print(f"\nReporte por clase - {split_name}")
    print(report_txt)

    metrics_dict = {
        "split": split_name,
        "accuracy": acc,
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,
        "f1_weighted": f1_w,
        "precision_macro": precision_m,
        "recall_macro": recall_m,
        "f1_macro": f1_m,
        "balanced_accuracy": bal_acc
    }

    return metrics_dict, report_txt


def save_confusion_matrix(y_true, y_pred, split_name, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)

    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation=45, colorbar=False)
    plt.title(f"Matriz de confusión - {split_name}")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("Guardado:", save_path)


def save_metrics_table_png(metrics_df, save_path):
    df_show = metrics_df.copy().round(4)

    fig, ax = plt.subplots(figsize=(12, 2 + 0.5 * len(df_show)))
    ax.axis("off")

    table = ax.table(
        cellText=df_show.values,
        colLabels=df_show.columns,
        loc="center",
        cellLoc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)

    plt.title("Resumen de métricas - LinearSVC", pad=20)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("Guardado:", save_path)

In [ ]:
param_grid = {
    "pca__n_components": [0.95, 256],
    "pca__whiten": [False],
    "svm__C": [0.01, 0.1, 1, 10, 50]
}

all_combinations = list(product(
    param_grid["pca__n_components"],
    param_grid["pca__whiten"],
    param_grid["svm__C"]
))

print(f"Total de combinaciones a evaluar: {len(all_combinations)}")

results = []
best_score = -1
best_params = None
best_model = None

for i, (n_components, whiten, C) in enumerate(all_combinations, start=1):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(
            n_components=n_components,
            whiten=whiten,
            random_state=RANDOM_STATE
        )),
        ("svm", LinearSVC(
            C=C,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            max_iter=10000
        ))
    ])

    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)

    acc_val = accuracy_score(y_val, y_val_pred)
    f1_w = f1_score(y_val, y_val_pred, average="weighted", zero_division=0)
    f1_m = f1_score(y_val, y_val_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_val, y_val_pred)

    results.append({
        "pca__n_components": n_components,
        "pca__whiten": whiten,
        "C": C,
        "accuracy_val": acc_val,
        "f1_weighted_val": f1_w,
        "f1_macro_val": f1_m,
        "balanced_acc_val": bal_acc
    })

    print(
        f"[{i}/{len(all_combinations)}] "
        f"pca__n_components={n_components}, pca__whiten={whiten}, C={C} "
        f"--> F1 macro val = {f1_m:.4f}"
    )

    if f1_m > best_score:
        best_score = f1_m
        best_params = {
            "pca__n_components": n_components,
            "pca__whiten": whiten,
            "C": C
        }
        best_model = model

print("\nMejores hiperparámetros encontrados:")
print(best_params)
print(f"Mejor F1 macro en validación: {best_score:.4f}")

results_df = pd.DataFrame(results).sort_values(by="f1_macro_val", ascending=False)
results_df = results_df.reset_index(drop=True)

print("\nTop 10 resultados:")
print(results_df.head(10))

results_df.to_csv(RESULTS_CSV_PATH, index=False)
print("Archivo guardado:", RESULTS_CSV_PATH)

In [ ]:
svm = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(
        n_components=best_params["pca__n_components"],
        whiten=best_params["pca__whiten"],
        random_state=RANDOM_STATE
    )),
    ("svm", LinearSVC(
        C=best_params["C"],
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=10000
    ))
])

print("\nEntrenando LinearSVC final...")
svm.fit(X_train, y_train)

# predicciones
y_train_pred = svm.predict(X_train)
y_val_pred   = svm.predict(X_val)
y_test_pred  = svm.predict(X_test)

# evaluación
train_metrics, train_report = evaluate_model(y_train, y_train_pred, "Train")
val_metrics, val_report     = evaluate_model(y_val, y_val_pred, "Validación")
test_metrics, test_report   = evaluate_model(y_test, y_test_pred, "Test")

metrics_df = pd.DataFrame([train_metrics, val_metrics, test_metrics])

print("\nResumen de métricas:")
print(metrics_df.round(4))

In [ ]:
save_confusion_matrix(y_val, y_val_pred, "Validación", CM_VAL_PATH)
save_confusion_matrix(y_test, y_test_pred, "Test", CM_TEST_PATH)

metrics_df.to_csv(METRICS_CSV_PATH, index=False)
save_metrics_table_png(metrics_df, METRICS_PNG_PATH)

with open(BEST_PARAMS_TXT_PATH, "w", encoding="utf-8") as f:
    f.write("MEJORES HIPERPARÁMETROS - LinearSVC\n")
    f.write("=" * 80 + "\n")
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nMejor F1 macro en validación: {best_score:.6f}\n")

with open(METRICS_TXT_PATH, "w", encoding="utf-8") as f:
    f.write("REPORTE COMPLETO - LinearSVC + Scaler + PCA\n")
    f.write("=" * 80 + "\n\n")

    f.write("Shapes\n")
    f.write("-" * 80 + "\n")
    f.write(f"X_train: {X_train.shape} | y_train: {y_train.shape}\n")
    f.write(f"X_val:   {X_val.shape} | y_val:   {y_val.shape}\n")
    f.write(f"X_test:  {X_test.shape} | y_test:  {y_test.shape}\n\n")

    f.write("Mejores hiperparámetros\n")
    f.write("-" * 80 + "\n")
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nMejor F1 macro en validación: {best_score:.6f}\n\n")

    f.write("Resumen numérico de métricas\n")
    f.write("-" * 80 + "\n")
    f.write(metrics_df.to_string(index=False))
    f.write("\n\n")

    f.write("Classification Report - Train\n")
    f.write("-" * 80 + "\n")
    f.write(train_report)
    f.write("\n\n")

    f.write("Classification Report - Validación\n")
    f.write("-" * 80 + "\n")
    f.write(val_report)
    f.write("\n\n")

    f.write("Classification Report - Test\n")
    f.write("-" * 80 + "\n")
    f.write(test_report)
    f.write("\n\n")

joblib.dump(svm, MODEL_PATH)

print("\nArchivos guardados correctamente:")
print(RESULTS_CSV_PATH)
print(BEST_PARAMS_TXT_PATH)
print(MODEL_PATH)
print(METRICS_CSV_PATH)
print(METRICS_TXT_PATH)
print(METRICS_PNG_PATH)
print(CM_VAL_PATH)
print(CM_TEST_PATH)

In [ ]:
print("Mejores hiperparámetros finales:")
print(best_params)

print("\nTop 10 combinaciones por F1 macro:")
print(results_df.head(10))